# S15 · Exponential smoothing (simple, Holt, Holt-Winters)

A second, gentler way to forecast the same kind of monthly demand series. The idea
in one line: average the past, but trust recent points more. We meet the three
members of the family and see how the weights fade smoothly into the past, and
which shape of data each member is meant for.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press play on each cell,
  top to bottom, and read the plain-English note above each one.
- New to time series? Open the primer
  `primers/time_series_and_forecasting.md` for a ten-minute, picture-first version.
- Already confident? Look for the cell marked **Stretch (optional)**.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses statsmodels for the smoothing models.
import sys
if "google.colab" in sys.modules:
    !pip install -q statsmodels
else:
    print("Not on Colab - assuming the libraries are already installed.")

In [ ]:
import numpy as np                     # fast maths on lists of numbers
import pandas as pd                      # tables and dated series
import matplotlib.pyplot as plt          # drawing charts

## The idea in one line

Exponential smoothing forecasts with a **weighted average of past observations**,
where the weights **fade exponentially**: the most recent point matters most, and
each older point counts a little less.

One knob, `alpha` (between 0 and 1), controls how fast the memory fades. There are
no stationarity worries and no differencing to do. It is much less fuss than
ARIMA, and a surprisingly strong baseline.

## Step 1 — picture the fading weights

Before any data, let us *see* what "exponentially fading weights" means. With
`alpha = 0.4`, the most recent point gets weight 0.4, the one before it gets
`0.4 x 0.6`, the next `0.4 x 0.6^2`, and so on. The weights shrink fast but never
quite reach zero.

In [ ]:
# Set a seed so anything random is reproducible.
np.random.seed(0)

alpha = 0.4
steps_back = np.arange(0, 14)               # 0 = most recent, 13 = far in the past

# The weight given to a point that is k steps back in time.
weights = alpha * (1 - alpha) ** steps_back

plt.figure(figsize=(8, 4))
plt.bar(steps_back, weights, color="#2E75B6")
plt.xlabel("how many steps back in time (0 = most recent)")
plt.ylabel("weight given to that point")
plt.title("Exponential smoothing: recent past weighted most, old past fades")
plt.show()

print("First five weights:", weights[:5].round(3))

## Step 2 — build a series with a trend and a season

We make a clear monthly demand series so the three smoothing methods have
something to grab onto: an upward trend plus a yearly (12-month) season.

In [ ]:
number_of_months = 96
time = np.arange(number_of_months)

# Trend + season + a little noise, added together.
trend = 40 + 0.5 * time
season = 8 * np.sin(2 * np.pi * time / 12)
noise = np.random.normal(0, 1.8, number_of_months)

monthly_sales = trend + season + noise

dates = pd.date_range("2012-01-01", periods=number_of_months, freq="MS")
series = pd.Series(monthly_sales, index=dates)

plt.figure(figsize=(9, 4))
plt.plot(series.index, series.values, color="#2E75B6")
plt.xlabel("month")
plt.ylabel("units sold")
plt.title("A monthly series with trend and season")
plt.show()

## Step 3 — Simple Exponential Smoothing (level only)

The simplest member tracks only the **level**, roughly "where the series is right
now". It has no idea about trend or season, so its forecast is a flat line. It is
the right tool only for a series that wobbles around a steady level, with no drift
and no repeating pattern.

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

# Fit simple exponential smoothing. It estimates the best alpha for us.
simple_model = SimpleExpSmoothing(series).fit()

# Forecast the next 24 months (it will be flat).
simple_forecast = simple_model.forecast(24)

plt.figure(figsize=(10, 5))
plt.plot(series.index, series.values, color="#2E75B6", label="history")
plt.plot(simple_forecast.index, simple_forecast.values,
         color="#C0392B", label="simple ES forecast")
plt.xlabel("month")
plt.ylabel("units sold")
plt.title("Simple ES: tracks only the level, so the forecast is flat")
plt.legend()
plt.show()

## Step 4 — Holt's method (level + trend)

Holt adds a second component, the **trend**, so the forecast can slope up or down
instead of staying flat. This suits a series that drifts but has no repeating
season, for example a product with steady growth and no strong seasonal swing.

In [ ]:
from statsmodels.tsa.holtwinters import Holt

# Holt adds a trend term to the level term.
holt_model = Holt(series).fit()

holt_forecast = holt_model.forecast(24)

plt.figure(figsize=(10, 5))
plt.plot(series.index, series.values, color="#2E75B6", label="history")
plt.plot(holt_forecast.index, holt_forecast.values,
         color="#C0392B", label="Holt forecast")
plt.xlabel("month")
plt.ylabel("units sold")
plt.title("Holt: level + trend, so the forecast can slope")
plt.legend()
plt.show()

## Step 5 — Holt-Winters (level + trend + season)

The full method adds a **seasonal** component, so the forecast keeps the slope
*and* reproduces the repeating yearly hump. This is the workhorse for clean
seasonal business data, like monthly retail sales. We tell it the season length is
12 months.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# trend="add" adds a trend; seasonal="add" adds a season of length 12.
holt_winters_model = ExponentialSmoothing(
    series,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
).fit()

# The fitted values show how well it tracks the history it learned from.
fitted_values = holt_winters_model.fittedvalues
holt_winters_forecast = holt_winters_model.forecast(24)

plt.figure(figsize=(10, 5))
plt.plot(series.index, series.values, color="#2E75B6", label="history")
plt.plot(fitted_values.index, fitted_values.values,
         color="#27AE60", label="Holt-Winters fit")
plt.plot(holt_winters_forecast.index, holt_winters_forecast.values,
         color="#C0392B", label="Holt-Winters forecast")
plt.xlabel("month")
plt.ylabel("units sold")
plt.title("Holt-Winters: keeps the trend AND repeats the season")
plt.legend()
plt.show()

## Step 6 — which method matched the data best?

Our series had both a trend and a season, so we expect Holt-Winters to fit best.
We compare how well each model tracked the history it learned from, using the
average squared error. Smaller is better.

This is the one rule to carry away: your method needs a component for every
structure you can see. A level-only model cannot follow a trend, and a
trend-only model cannot follow a season.

In [ ]:
# Compare the fit on the training history for each model.
# We use the mean squared error: the average of (actual - fitted) squared.
def mean_squared_error(actual, fitted):
    difference = actual - fitted
    return np.mean(difference ** 2)

simple_error = mean_squared_error(series.values, simple_model.fittedvalues.values)
holt_error = mean_squared_error(series.values, holt_model.fittedvalues.values)
holt_winters_error = mean_squared_error(series.values,
                                        holt_winters_model.fittedvalues.values)

print("In-sample mean squared error (smaller is better):")
print("  Simple ES    :", round(simple_error, 2))
print("  Holt         :", round(holt_error, 2))
print("  Holt-Winters :", round(holt_winters_error, 2))
print()
print("Holt-Winters should win here, because our data has BOTH trend and season.")

### Stretch (optional) — smooth it yourself with a loop

Skip this if you are new to code. If you are comfortable, here is the whole of
simple exponential smoothing in one small loop, so the "weighted average of the
past" is not a black box. The recipe is: each new smoothed value is
`alpha` times today's actual, plus `(1 - alpha)` times yesterday's smoothed value.
Roll that forward and you have already averaged the whole past with fading
weights, without ever storing them.

In [ ]:
alpha = 0.4
actual_values = series.values

# Start the smoothed track at the first actual value.
smoothed = [actual_values[0]]

# Walk forward one step at a time, blending today with the running memory.
for step in range(1, len(actual_values)):
    todays_value = actual_values[step]
    yesterdays_smoothed = smoothed[step - 1]
    new_smoothed = alpha * todays_value + (1 - alpha) * yesterdays_smoothed
    smoothed.append(new_smoothed)

print("first 5 actual  :", actual_values[:5].round(1))
print("first 5 smoothed:", np.array(smoothed[:5]).round(1))
print()
print("That one blend line IS exponential smoothing - the fading weights are hidden inside it.")

## What you just did

You met the exponential-smoothing family and the rule that ties it to the parts of
a series: your method needs a component for every structure you see. Simple ES
handles a level, Holt adds a trend, Holt-Winters adds a season. All three rest on
the same idea as ARIMA, a weighted average of the past, only here the weights fade
in a fixed exponential pattern instead of being fitted.

Next notebook: `03_nifty_arima_vs_smoothing.ipynb`, the lab, where ARIMA and
exponential smoothing race on the same real series.